# MPS：GPU 为什么能共享

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rootsongjc/gpu-lab-notebooks/blob/main/labs/zh/lab-09-mps.ipynb)

动手理解多进程共享 GPU。先用两个独立进程观察默认的 time-slicing 行为与每个进程的 context 开销，再尝试启动 CUDA MPS 让 kernel 并发，并讲清 MPS 解决了什么、没解决什么，以及 Kubernetes 上的共享方案怎样补上它的缺口。

本 notebook 配合教程章节 [MPS：GPU 为什么能共享](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-09-mps/) 使用。运行前请确认 **Runtime → Change runtime type → T4 GPU**。

每个 cell 对应教程中的一步；Markdown 单元摘录了必要的解释，完整上下文请回到教程阅读。

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv

## 为什么这个问题重要

[Lab 08](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-08-cuda-kernel/) 的 kernel 是单进程独占整张 GPU。但真实平台里，推理服务常常一个进程只用 30% 算力，独占整卡太浪费。**GPU 共享**就是“让多个进程/容器在同一张卡上合理共存”。MPS 是这条路上绕不开的基础概念，但它只解决了“并发”。理解它缺什么，你才能看懂 Kubernetes 上各种共享方案在补什么。

## 三种“共存”机制的底层区别

![三种共存机制：默认 time-slicing 下多个 context 轮流独占 GPU、MPS 下多进程 kernel 并发、软件配额方案在 CUDA API 层按配额节流](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-09-mps/gpu-sharing-three-modes.svg)



| 机制 | 并发方式 | 显存 | 故障隔离 | 配额 |
| :--- | :--- | :--- | :--- | :--- |
| 默认 time-slicing（多 context） | 驱动按时间片轮流让一个 context 独占 SM；kernel **不并发** | 每进程独立 context，各自占用，**无上限** | 一个进程崩溃不影响其它 | 无 |
| **MPS** | 多个进程的 kernel 经由 MPS server **同时**在 SM 上执行 | 每个客户端仍各自占用显存（Volta+），另加一个 server 进程 | 弱：Volta+ 有按客户端的地址空间隔离，但 server 故障影响全部 | 有但粗糙：环境变量 `CUDA_MPS_ACTIVE_THREAD_PERCENTAGE`、`CUDA_MPS_PINNED_DEVICE_MEM_LIMIT`，启动时设定，无调度器参与 |
| **软件配额（以 HAMi-core 为例）** | 不改变驱动调度；在用户态 **拦截 CUDA API**，按配额节流 kernel 发射、拒绝超额 `cudaMalloc` | 每容器显存硬上限 | 进程级，与 time-slicing 相同 | 显存与算力份额，由 Kubernetes 调度器分配 |

## 第 1 步：两个进程，各自的 context

先写一个会持续做矩阵乘法的小脚本：

In [ ]:
%%writefile worker.py
import sys, time, torch
name, seconds = sys.argv[1], float(sys.argv[2])
a = torch.randn(4096, 4096, device="cuda"); b = torch.randn(4096, 4096, device="cuda")
torch.cuda.synchronize()
t0 = time.time(); n = 0
while time.time() - t0 < seconds:
    c = a @ b; n += 1
    if n % 10 == 0:
        torch.cuda.synchronize()   # 等 GPU 真正算完再继续，否则只是往队列里塞任务，计数会虚高
torch.cuda.synchronize()
dt = time.time() - t0
print(f"{name}: {n} matmuls in {dt:.1f}s -> {n/dt:.1f} it/s", flush=True)

脚本持续运行指定秒数，最后打印每秒完成多少次矩阵乘法（`it/s`），这就是它的吞吐。先单独跑一个，记下基线：

In [ ]:
!python worker.py solo 10

实测输出：

```text
solo: 310 matmuls in 10.1s -> 30.6 it/s
```

一个进程独占 T4 时，每秒能完成约 30 次 4096×4096 的矩阵乘法。

然后同时跑两个，并在它们运行期间看一眼显存和进程列表：

In [ ]:
import subprocess, time

def start(script, name, seconds, env=None):
    # 用管道接住输出：Colab 不会显示后台子进程直接打印的内容
    return subprocess.Popen(["python", script, name, str(seconds)], stdout=subprocess.PIPE, text=True, env=env)

p1 = start("worker.py", "A", 10); p2 = start("worker.py", "B", 10)
time.sleep(6)
!nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv,noheader
!nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv,noheader
for p in (p1, p2):
    print(p.communicate()[0].strip())

实测输出（PID 每次不同）：

```text
16095, python3, 390 MiB
16096, python3, 390 MiB
783 MiB, 100 %
A: 140 matmuls in 10.3s -> 13.6 it/s
B: 140 matmuls in 10.3s -> 13.6 it/s
```

逐段解读：

- 进程列表里有两个 `python3`，各占 390 MiB：其中约 120 MiB 是 CUDA context（[Lab 04](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-04-where-did-vram-go/) 测过），192 MiB 是 a、b、c 三个矩阵，剩下的是 cuBLAS 的工作区。**每个进程都付了一份 context 的入场费**；
- GPU 利用率 100%，显存合计 783 MiB；
- A 和 B 各 13.6 it/s，合计 27.2 it/s，只有 solo 的 89%。GPU 在两个进程之间轮流切换，每人分到不到一半，**切换本身损失了约 11% 的吞吐**。

这就是默认的 time-slicing 模式：**公平但不并发，隔离但无配额**。如果 worker B 把矩阵改成 12288×12288、占用 10 GB 显存，A 就会 OOM，没有任何机制阻止它。

## 第 2 步：换成小矩阵再看一次

推理服务的特点是 kernel 又小又多。把矩阵改成 `512×512`，重复第 1 步：

In [ ]:
!sed 's/4096/512/g' worker.py > worker_small.py
!python worker_small.py solo 10
p1 = start("worker_small.py", "A", 10); p2 = start("worker_small.py", "B", 10)
for p in (p1, p2):
    print(p.communicate()[0].strip())

实测输出：

```text
solo: 145040 matmuls in 10.0s -> 14503.5 it/s
A: 64680 matmuls in 10.0s -> 6467.9 it/s
B: 64460 matmuls in 10.0s -> 6444.5 it/s
```

小矩阵每次只要几十微秒，所以每秒能做一万多次。两个进程合计 12912 it/s，同样只有 solo 的 89%。在这张 T4 上，大小矩阵的切换损耗差不多；进程越多、切换越频繁，这笔代价越大。能不能让两个进程的 kernel **同时**跑、省掉切换？这就是 MPS 的动机。

## 第 3 步：启动 MPS

MPS 的两个可执行文件 `nvidia-cuda-mps-control` 和 `nvidia-cuda-mps-server` **随驱动安装**，不属于 CUDA Toolkit：

In [ ]:
!which nvidia-cuda-mps-control nvidia-cuda-mps-server || echo "MPS binaries not in this image"

实测输出：

```text
/opt/bin/nvidia-cuda-mps-control
/opt/bin/nvidia-cuda-mps-server
```

Colab 镜像里两个文件都在。以当前用户启动 MPS 的控制进程（daemon）：

In [ ]:
import os
os.environ["CUDA_MPS_PIPE_DIRECTORY"] = "/tmp/nvidia-mps"
os.environ["CUDA_MPS_LOG_DIRECTORY"]  = "/tmp/nvidia-log"
!mkdir -p /tmp/nvidia-mps /tmp/nvidia-log && nvidia-cuda-mps-control -d && sleep 2 && echo "MPS control daemon started"
!echo get_server_list | nvidia-cuda-mps-control

实测输出：

```text
MPS control daemon started
```

`get_server_list` 这一行没有输出是正常的：control daemon 只负责接待，真正干活的 MPS server 要等第一个 CUDA 程序连上来才启动。两个环境变量告诉之后启动的 CUDA 程序去哪里找 daemon。

现在把第 2 步的小矩阵 worker 接进 MPS：

In [ ]:
p1 = start("worker_small.py", "A-mps", 10, env=os.environ); p2 = start("worker_small.py", "B-mps", 10, env=os.environ)
time.sleep(6)
!nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv,noheader
for p in (p1, p2):
    print(p.communicate()[0].strip())
!echo get_server_list | nvidia-cuda-mps-control
!echo quit | nvidia-cuda-mps-control

实测输出：

```text
16348, nvidia-cuda-mps-server, 28 MiB
16328, python3, 136 MiB
16327, python3, 136 MiB
A-mps: 66870 matmuls in 10.0s -> 6686.9 it/s
B-mps: 65748 matmuls in 10.0s -> 6574.2 it/s
16348
```

逐段解读：

- 进程列表多了一个 `nvidia-cuda-mps-server`，`get_server_list` 打印的 16348 就是它的 PID：**MPS 确实接管了**，两个 python 进程的 kernel 都经由这个 server 提交；
- 两个 python 进程仍然各占约 136 MiB，和不开 MPS 时（实测 138 MiB）几乎一样。在 Volta 及以后的 GPU 上，每个 MPS 客户端仍有自己独立的显存空间，**MPS 并不能省下每个进程的那份 context**；
- 合计吞吐 13261 it/s，是 solo 的 91%，只比第 2 步的 89% 高一点。用 4096 的大矩阵重复一遍，结果也一样（实测合计 28.3 it/s，solo 的 92%）。

把三步的实测数据放在一起：

![左侧是两进程合计吞吐相对单进程的比例，轮流约 89%、MPS 约 91% 到 92%；右侧是每个 worker 进程 390 MiB 显存的构成：context、三个矩阵和 cuBLAS 工作区](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-09-mps/mps-measured.svg)

为什么 MPS 几乎没提速？因为第 1、2 步里 GPU 利用率已经是 100%：一个进程就能把 SM 填满，MPS 没有空闲的 SM 可以分给另一个进程，能省的只有切换开销那一点。**MPS 真正有用的场景，是每个进程单独都用不满 GPU 的时候**，比如小批量推理服务。

> **Colab 里能起来，不代表 Kubernetes 上也简单**
>
> Colab 的 runtime 是一个容器，MPS 在这里能以普通用户启动（2026-11-08 实测）。但在 Kubernetes 上，MPS daemon 必须作为节点级 DaemonSet 运行，所有 Pod 要挂同一个管道目录，客户端崩溃还可能拖垮 server，影响同卡的其它 Pod。NVIDIA device plugin 从 v0.15 起提供了 MPS 模式来封装这些细节，由平台层替你管理 daemon。带着这个理解去读 [用 GPU Operator 配置 GPU 共享](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/hands-on/gpu-operator-sharing/)。

## 平台层还要补什么

MPS 解决了“并发”，但在多租户平台视角下还缺三样东西：

1. **调度器不知情**：`CUDA_MPS_ACTIVE_THREAD_PERCENTAGE` 是进程启动时的环境变量，Kubernetes 调度器无法据此决定“这个 Pod 该去哪个节点、还剩多少额度”。平台需要的是调度器可见的资源量。
2. **显存配额粗糙**：`CUDA_MPS_PINNED_DEVICE_MEM_LIMIT` 只限制 pinned 显存，且同样是静态环境变量，平台需要按 [Lab 04](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-04-where-did-vram-go/) 里 `nvidia-smi` 的统计方式给每个容器设显存上限。
3. **运维耦合**：节点常驻特权 daemon、共享管道目录、server 单点故障，都与容器“各自独立、随时重建”的生命周期相冲突。

Kubernetes 上的共享方案就是从不同角度补这三样东西：

- **GPU Operator 的 MPS 与 time-slicing 模式**：把一张卡上报成多个副本，由平台管理 MPS daemon，但没有按 Pod 的显存上限。
- **HAMi**：在用户态拦截 CUDA API，给每个容器加调度器可见的显存与算力配额，详见 [HAMi 软件切分与配额](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/hands-on/hami-sharing/)。
- **KAI Scheduler 的 GPU 分数**：由调度器按比例记账，CUDA 13.4 起可由驱动强制显存上限，详见 [KAI GPU 分数与显存强制共享](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/hands-on/kai-gpu-fraction/)。
- **MIG**：在支持的卡上直接做硬件切分，下一章展开。

## 验证

完成后你应该能：

- 说出默认 time-slicing、MPS、软件配额三者在并发 / 显存 / 隔离 / 配额四个维度上的差别；
- 用第 1 步的数据解释“每个进程都付 context 入场费”和“公平但不并发”；
- 用第 1、2 步的数据说出 time-slicing 的切换损耗大约是多少；
- 用第 3 步的数据解释为什么 GPU 已满载时 MPS 几乎不提速，以及在 Kubernetes 里为什么难部署；
- 说明平台层在 MPS 之上还要补哪三样东西。

## 故障模式



| 症状 | 原因 | 处理 |
| :--- | :--- | :--- |
| A / B 的结果没有打印出来 | Colab 不显示后台子进程直接打印的内容 | 用 `start()` 里的 `stdout=subprocess.PIPE` 接住，再 `communicate()` 打印 |
| `nvidia-cuda-mps-control: command not found` | 容器镜像未包含驱动的 MPS 二进制 | 记录“不可用”，继续第 1–2 步的结论 |
| 刚启动 daemon 时 `get_server_list` 为空 | server 要等第一个客户端连上才启动 | 正常，跑完 worker 后再查一次 |
| 开了 MPS 吞吐几乎不变 | GPU 已经满载，没有空闲 SM 可分 | 正常，这正是第 3 步要观察的结论 |
| 两个 worker 都报 `CUDA out of memory` | 显存被其它 Lab 的残留进程占满 | 不应发生（实测两个 worker 合计约 800 MiB）；若发生，先 Restart session |
| `--query-compute-apps` 为空 | 容器 PID 命名空间（Colab 上能看到，K8s 普通容器里常见） | 正常，用 `memory.used` 的增量推断 |
| Colab session 超时 | 免费版时长限制 | 拆小实验，见 [篇首页](../#colab-的能用多久) |

## 跑完之后

如果你看到的输出与教程里的预期明显不符，或者碰到了教程没有列出的故障模式，欢迎带上驱动 / CUDA 版本和关键输出在 [rootsongjc/gpu-lab-notebooks](https://github.com/rootsongjc/gpu-lab-notebooks) 提 Issue 或 PR。